# Laboratório 07 — Especialização de LLMs com LoRA e QLoRA
**Domínio:** Previsões Climáticas e Meteorologia  
**Modelo:** `meta-llama/Llama-2-7b-hf`  
**Ambiente:** Google Colab (T4 / A100)

> ⚠️ Antes de iniciar: vá em **Runtime → Change runtime type** e selecione **GPU** (T4 ou A100).

## 0. Instalação de Dependências

In [ ]:
!pip install -q \
    transformers==4.40.0 \
    peft==0.10.0 \
    trl==0.8.6 \
    bitsandbytes==0.43.1 \
    datasets==2.19.0 \
    accelerate==0.29.3 \
    openai==1.25.0 \
    scipy

print('✅ Dependências instaladas.')

## 0.1 Verificação da GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ GPU não detectada! Mude o runtime para GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')

## 0.2 Autenticação HuggingFace (necessário para Llama 2)

In [ ]:
import os
from huggingface_hub import login

# Cole seu token do HuggingFace (https://huggingface.co/settings/tokens)
HF_TOKEN = "hf_SEU_TOKEN_AQUI"
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print('✅ Autenticado no HuggingFace.')

---
## Passo 1 — Geração do Dataset Sintético
**Domínio:** Previsões Climáticas e Meteorologia  
**API:** OpenAI GPT-3.5-turbo  
**Formato:** `.jsonl` — 90% treino / 10% teste  
**Mínimo:** 50 pares válidos (prompt / response)

In [ ]:
import json, random, time
from openai import OpenAI

# ── Configurações ──────────────────────────────────────────────────
OPENAI_API_KEY = "sk-SEU_TOKEN_OPENAI_AQUI"   # substitua pela sua chave
DOMAIN         = "previsões climáticas e meteorologia"
N_SAMPLES      = 60
TRAIN_RATIO    = 0.90
TRAIN_FILE     = "dataset_train.jsonl"
TEST_FILE      = "dataset_test.jsonl"

SYSTEM_PROMPT = f"""Você é um meteorologista especialista em {DOMAIN}.
Gere EXATAMENTE um par JSON com as chaves "prompt" e "response".
- "prompt": pergunta ou instrução técnica clara sobre {DOMAIN} (em português).
- "response": resposta detalhada, correta e didática (em português), com dados e explicações científicas.
Retorne APENAS o JSON puro, sem texto adicional, sem blocos de markdown."""

SEED_TOPICS = [
    "fenômenos El Niño e La Niña e seus impactos no Brasil",
    "modelos numéricos de previsão do tempo (GFS, ECMWF)",
    "diferença entre tempo e clima",
    "formação e classificação de nuvens",
    "ciclones extratropicais e tropicais",
    "chuvas convectivas versus estratiformes",
    "índices de instabilidade atmosférica (CAPE, LI)",
    "frentes frias e frentes quentes",
    "umidade relativa do ar e ponto de orvalho",
    "inversão térmica e poluição atmosférica",
    "monções e ventos alísios",
    "mudanças climáticas e aquecimento global",
    "ilhas de calor urbanas",
    "precipitação e ciclo hidrológico",
    "radiação solar e balanço de energia na Terra",
    "pressão atmosférica e isobares",
    "tornados: formação e escala Fujita",
    "granizo: formação e condições necessárias",
    "seca meteorológica versus seca hidrológica",
    "satélites meteorológicos e interpretação de imagens",
    "radar meteorológico e leitura de refletividade",
    "índice de conforto térmico e sensação térmica",
    "chuva ácida: causas e efeitos",
    "ondas de calor e ondas de frio",
    "ZCIT (Zona de Convergência Intertropical)",
]

def generate_pair(client, topic):
    try:
        resp = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"Crie um par instrução/resposta técnica sobre: {topic}."}
            ],
            temperature=0.85,
            max_tokens=700,
        )
        raw = resp.choices[0].message.content.strip()
        raw = raw.removeprefix('```json').removeprefix('```').removesuffix('```').strip()
        pair = json.loads(raw)
        assert "prompt" in pair and "response" in pair
        assert len(pair["prompt"]) > 15 and len(pair["response"]) > 30
        return pair
    except Exception as e:
        print(f"  [WARN] {topic}: {e}")
        return None

# ── Geração ────────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)
pairs  = []
topics = (SEED_TOPICS * 3)[:N_SAMPLES]
random.shuffle(topics)

for i, topic in enumerate(topics):
    print(f"[{i+1:02d}/{N_SAMPLES}] {topic}")
    p = generate_pair(client, topic)
    if p:
        pairs.append(p)
    time.sleep(0.6)

assert len(pairs) >= 50, f"Apenas {len(pairs)} pares. Mínimo exigido: 50."
random.shuffle(pairs)

# ── Divisão 90 / 10 ────────────────────────────────────────────────
split       = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:split]
test_pairs  = pairs[split:]

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_jsonl(train_pairs, TRAIN_FILE)
save_jsonl(test_pairs,  TEST_FILE)

print(f"\n✅ Dataset gerado!")
print(f"   Treino : {len(train_pairs)} exemplos → {TRAIN_FILE}")
print(f"   Teste  : {len(test_pairs)} exemplos → {TEST_FILE}")

---
## Passo 2 — Configuração da Quantização (QLoRA)
Carrega o modelo em **4-bits** com tipo `nf4` (NormalFloat 4-bit) e `compute_dtype=float16`.

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                   # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.float16,         # compute dtype: float16
    bnb_4bit_use_double_quant=True,
)

print("✅ BitsAndBytesConfig pronto:")
print(f"   load_in_4bit          = True")
print(f"   bnb_4bit_quant_type   = nf4")
print(f"   bnb_4bit_compute_dtype = float16")
print(f"   bnb_4bit_use_double_quant = True")

---
## Passo 3 — Arquitetura LoRA
Congela os pesos originais e treina apenas as matrizes de decomposição.

| Hiperparâmetro | Valor | Descrição |
|---|---|---|
| `r` (Rank) | **64** | Dimensão das matrizes menores |
| `lora_alpha` | **16** | Fator de escala dos novos pesos |
| `lora_dropout` | **0.1** | Evita overfitting |
| `task_type` | `CAUSAL_LM` | Modelagem de linguagem causal |

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=64,                           # Rank: dimensão das matrizes menores
    lora_alpha=16,                  # Alpha: fator de escala dos novos pesos
    lora_dropout=0.1,               # Dropout: evita overfitting
    bias="none",
    task_type=TaskType.CAUSAL_LM,   # Tarefa: modelagem de linguagem causal
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

print("✅ LoraConfig pronto:")
print(f"   r            = {lora_config.r}")
print(f"   lora_alpha   = {lora_config.lora_alpha}")
print(f"   lora_dropout = {lora_config.lora_dropout}")
print(f"   task_type    = {lora_config.task_type}")

---
## Passo 4 — Carregamento do Modelo e Pipeline de Treinamento

| Argumento | Valor | Motivo |
|---|---|---|
| `optim` | `paged_adamw_32bit` | Transfere picos de memória GPU → CPU |
| `lr_scheduler_type` | `cosine` | LR decai suavemente em curva cosseno |
| `warmup_ratio` | `0.03` | Primeiros 3% do treino: LR sobe gradualmente |

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "meta-llama/Llama-2-7b-hf"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, token=HF_TOKEN, trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# Modelo base quantizado
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

# Prepara para treinamento quantizado
model = prepare_model_for_kbit_training(model)

# Aplica LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Modelo com LoRA pronto para treinamento.")

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "./llama2-clima-qlora-adapter"

# ── Carrega e formata dataset ───────────────────────────────────────
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def format_instruction(ex):
    return {"text": f"### Instrução:\n{ex['prompt']}\n\n### Resposta:\n{ex['response']}"}

train_dataset = Dataset.from_list([format_instruction(x) for x in load_jsonl(TRAIN_FILE)])
test_dataset  = Dataset.from_list([format_instruction(x) for x in load_jsonl(TEST_FILE)])
print(f"Treino: {len(train_dataset)} | Teste: {len(test_dataset)}")

# ── TrainingArguments ──────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    # ─── Obrigatórios pelo laboratório ───────────────────────────
    optim="paged_adamw_32bit",           # AdamW paginado: picos GPU → CPU
    lr_scheduler_type="cosine",          # decaimento em curva cosseno
    warmup_ratio=0.03,                   # 3% de aquecimento inicial
    # ─── Demais ──────────────────────────────────────────────────
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    evaluation_strategy="epoch",
    group_by_length=True,
    gradient_checkpointing=True,         # economiza VRAM no Colab
    report_to="none",
)

# ── SFTTrainer ─────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

print("✅ Trainer configurado. Iniciando treinamento...")
trainer.train()

In [ ]:
# Salva o adaptador LoRA (obrigatório — Passo 4)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Adaptador LoRA salvo em: {OUTPUT_DIR}")

---
## Inferência — Testando o modelo fine-tunado

In [ ]:
from peft import PeftModel

# Recarrega modelo + adaptador para inferência
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map="auto", token=HF_TOKEN,
)
inf_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
inf_model.eval()

def gerar_resposta(instrucao, max_new_tokens=250):
    prompt = f"### Instrução:\n{instrucao}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(inf_model.device)
    with torch.no_grad():
        out = inf_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("### Resposta:")[-1].strip()

perguntas = [
    "O que é El Niño e como ele afeta as chuvas no Nordeste brasileiro?",
    "Explique o que é CAPE e como esse índice é usado na previsão de tempestades.",
    "Qual a diferença entre frente fria e frente quente?",
]

for pergunta in perguntas:
    print("\n" + "="*60)
    print(f"INSTRUÇÃO: {pergunta}")
    print("-"*60)
    print(f"RESPOSTA:\n{gerar_resposta(pergunta)}")